In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from typing import Dict, List

# =============================================================================
# 0) Configuration block (only modify here)
# =============================================================================
PLOT_CFG = {
    # Per-facet size
    "height": 2,     # subplot height
    "aspect": 1.7,    # subplot width/height ratio
    # Line style
    "use_dashed": True,
    "linewidth": 2.0,
    "shade_alpha": 0.18,
    # Font / style
    "style": "whitegrid",
    "palette": "deep",
    "dpi": 150,
    # Legend position
    "legend_loc": "best",
    # Output filename
    "outfile": "result_1.pdf",
}

# Model name -> panel title
TITLE_MAP: Dict[str, str] = {
    'bioS_noise_0_base_250818-070540': "SINGLE",
    'bioS_noise_0_context_250818-095527': "REPEATED",
    'bioS_noise_0_multiple-context_250818-124537': "REPEATED + MIX"
}

# CSV column name -> legend label (LaTeX)
METRIC_MAP: Dict[str, str] = {
    'em/param': r'$\mathrm{Acc}_{\mathrm{PKU}}$',
    'em/multi_ood_in_ctx': r'$\mathrm{Acc}_{\mathrm{ICKU}}$',
    # 'em/pert_ctx_orig': r'$\mathrm{Pref}_{\mathrm{PK}}$',
    # 'em/pert_ctx_pert': r'$\mathrm{Pref}_{\mathrm{ICK}}$',
}

# Color / line style map (by label)
COLOR_MAP = {
    r'$\mathrm{Acc}_{\mathrm{PKU}}$': sns.color_palette("deep")[2],
    r'$\mathrm{Acc}_{\mathrm{ICKU}}$': sns.color_palette("deep")[0],
    r'$\mathrm{Pref}_{\mathrm{PK}}$': sns.color_palette("deep")[3],
    r'$\mathrm{Pref}_{\mathrm{ICK}}$': sns.color_palette("deep")[1],
}
LINESTYLE_MAP = {
    r'$\mathrm{Acc}_{\mathrm{PKU}}$': (2, 2),
    r'$\mathrm{Acc}_{\mathrm{ICKU}}$': (2, 2),
    r'$\mathrm{Pref}_{\mathrm{PK}}$': (8, 2, 2, 2),
    r'$\mathrm{Pref}_{\mathrm{ICK}}$': (4, 4),
}

# =============================================================================
# 1) Style setup (remove figure.figsize duplication: control size via FacetGrid only)
# =============================================================================
def setup_style() -> None:
    sns.set_theme(
        style=PLOT_CFG["style"],
        palette=PLOT_CFG["palette"],
        rc={
            "figure.dpi": PLOT_CFG["dpi"],
            "axes.titlesize": 15,
            "axes.labelsize": 13,
            "legend.fontsize": 15,
            "lines.linewidth": PLOT_CFG["linewidth"],
        }
    )

# =============================================================================
# 2) Data loading & statistics aggregation
# =============================================================================
def load_stats(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    # Rename only metrics of interest to labels, long-form (including seed)
    rename_cols = {k: v for k, v in METRIC_MAP.items() if k in df.columns}
    df = df.rename(columns=rename_cols)
    metrics = list(rename_cols.values())

    long_seed = pd.melt(
        df,
        id_vars=["model", "step", "seed"],
        value_vars=[c for c in metrics if c in df.columns],
        var_name="Metric",
        value_name="Accuracy"
    )
    if long_seed["Accuracy"].max() <= 1.0:
        long_seed["Accuracy"] *= 100.0

    long_seed["Model_Title"] = long_seed["model"].map(TITLE_MAP).fillna(long_seed["model"])

    stats = (
        long_seed
        .groupby(["model", "Model_Title", "step", "Metric"], as_index=False)
        .agg(Mean=("Accuracy", "mean"), Std=("Accuracy", "std"), N=("Accuracy", "count"))
        .sort_values(["Model_Title", "Metric", "step"])
        .reset_index(drop=True)
    )
    return stats

# =============================================================================
# 3) Plotting (mean + std shading)
# =============================================================================
def plot_with_std(stats_df: pd.DataFrame, models: List[str], outfile: str = None) -> None:
    model_titles = [TITLE_MAP.get(m, m) for m in models]
    data = stats_df[stats_df["Model_Title"].isin(model_titles)].copy()
    if data.empty:
        print("No data found for specified models.")
        return

    g = sns.FacetGrid(
        data,
        col="Model_Title",
        col_order=model_titles,
        height=PLOT_CFG["height"],
        aspect=PLOT_CFG["aspect"],
        sharey=True,
        margin_titles=False
    )

    for ax, title in zip(g.axes.flat, model_titles):
        sub = data[data["Model_Title"] == title]
        for metric, mdf in sub.groupby("Metric", sort=False):
            mdf = mdf.sort_values("step")
            x = mdf["step"].to_numpy()
            y = mdf["Mean"].to_numpy()
            s = mdf["Std"].to_numpy()
            color = COLOR_MAP.get(metric, None)

            if PLOT_CFG["use_dashed"]:
                ax.plot(x, y, label=str(metric), color=color, dashes=LINESTYLE_MAP.get(metric, (6, 3)))
            else:
                ax.plot(x, y, label=str(metric), color=color)

            ax.fill_between(x, y - s, y + s, color=color, alpha=PLOT_CFG["shade_alpha"])

        ax.set_title(title, fontsize=15, weight="bold", style="italic")
        ax.set_xlabel("Training Steps", fontsize=13)
        ax.grid(True, alpha=0.3)

    g.axes.flat[0].set_ylabel("Accuracy (%)", fontsize=13)

    handles, labels = g.axes.flat[-1].get_legend_handles_labels()
    g.axes.flat[-1].legend(handles=handles, labels=labels, loc=PLOT_CFG["legend_loc"], frameon=True)

    out = outfile or PLOT_CFG["outfile"]
    g.fig.savefig(out, dpi=PLOT_CFG["dpi"], bbox_inches="tight", format="pdf")
    print(f"Figure saved as '{out}'")
    plt.show()

# =============================================================================
# 4) Execution
# =============================================================================
if __name__ == "__main__":
    setup_style()
    csv_path = "gpt2train/probe_results_multi.csv"
    stats_df = load_stats(csv_path)
    print(f"Stats: {stats_df.shape}")

    models_to_display = [
        'bioS_noise_0_base_250818-070540',
        'bioS_noise_0_context_250818-095527',
        'bioS_noise_0_multiple-context_250818-124537'
    ]
    plot_with_std(stats_df, models_to_display)

In [ ]:
### result 2
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from typing import Dict, List

# =============================================================================
# 0) Configuration block (only modify here)
# =============================================================================
PLOT_CFG = {
    # Per-facet size
    "height": 2,     # subplot height
    "aspect": 1.7,     # subplot width/height ratio
    # Line style
    "use_dashed": True,
    "linewidth": 2.0,
    "shade_alpha": 0.18,
    # Font / style
    "style": "whitegrid",
    "palette": "deep",
    "dpi": 150,
    # Legend position
    "legend_loc": "best",
    # Output filename
    "outfile": "result_2.pdf",
}

# Model name -> panel title (noise version)
TITLE_MAP: Dict[str, str] = {
    'bioS_noise_0_multiple-context_250818-124537': "No noise",
    'bioS_noise_1_multiple-context_passage-noise_250816-095848': "1% noise",
    'bioS_noise_5_multiple-context_passage-noise_250816-125421': "5% noise",
}

# CSV column name -> legend label (LaTeX)
METRIC_MAP: Dict[str, str] = {
    'em/param': r'$\mathrm{Acc}_{\mathrm{PKU}}$',
    'em/multi_ood_in_ctx': r'$\mathrm{Acc}_{\mathrm{ICKU}}$',
    'em/pert_ctx_orig': r'$\mathrm{Pref}_{\mathrm{PK}}$',
    'em/pert_ctx_pert': r'$\mathrm{Pref}_{\mathrm{ICK}}$',
}

# Color / line style map (by label)
COLOR_MAP = {
    r'$\mathrm{Acc}_{\mathrm{PKU}}$': sns.color_palette("deep")[2],
    r'$\mathrm{Acc}_{\mathrm{ICKU}}$': sns.color_palette("deep")[0],
    r'$\mathrm{Pref}_{\mathrm{PK}}$': sns.color_palette("deep")[3],
    r'$\mathrm{Pref}_{\mathrm{ICK}}$': sns.color_palette("deep")[1],
}
LINESTYLE_MAP = {
    r'$\mathrm{Acc}_{\mathrm{PKU}}$': (2, 2),
    r'$\mathrm{Acc}_{\mathrm{ICKU}}$': (3, 1),
    r'$\mathrm{Pref}_{\mathrm{PK}}$': (1, 3),
    r'$\mathrm{Pref}_{\mathrm{ICK}}$': (1, 3),
}

# =============================================================================
# 1) Style setup (remove figure.figsize duplication: control size via FacetGrid only)
# =============================================================================
def setup_style() -> None:
    sns.set_theme(
        style=PLOT_CFG["style"],
        palette=PLOT_CFG["palette"],
        rc={
            "figure.dpi": PLOT_CFG["dpi"],
            "axes.titlesize": 14,
            "axes.labelsize": 12,
            "legend.fontsize": 11,
            "lines.linewidth": PLOT_CFG["linewidth"],
        }
    )

# =============================================================================
# 2) Data loading & statistics aggregation (per-seed values -> mean/std)
# =============================================================================
def load_stats(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    # Rename metrics of interest to labels
    rename_cols = {k: v for k, v in METRIC_MAP.items() if k in df.columns}
    df = df.rename(columns=rename_cols)
    metrics = list(rename_cols.values())

    # long-form (keep seed)
    long_seed = pd.melt(
        df,
        id_vars=["model", "step", "seed"],
        value_vars=[c for c in metrics if c in df.columns],
        var_name="Metric",
        value_name="Accuracy"
    )
    # Convert to percent if 0~1 scale
    if long_seed["Accuracy"].max() <= 1.0:
        long_seed["Accuracy"] *= 100.0

    # Model title mapping
    long_seed["Model_Title"] = long_seed["model"].map(TITLE_MAP).fillna(long_seed["model"])

    # mean/std per (model, step, Metric)
    stats = (
        long_seed
        .groupby(["model", "Model_Title", "step", "Metric"], as_index=False)
        .agg(Mean=("Accuracy", "mean"), Std=("Accuracy", "std"), N=("Accuracy", "count"))
        .sort_values(["Model_Title", "Metric", "step"])
        .reset_index(drop=True)
    )
    return stats

# =============================================================================
# 3) Plotting (mean + std shading)
# =============================================================================
def plot_with_std(stats_df: pd.DataFrame, models: List[str], outfile: str = None) -> None:
    model_titles = [TITLE_MAP.get(m, m) for m in models]
    data = stats_df[stats_df["Model_Title"].isin(model_titles)].copy()
    if data.empty:
        print("No data found for specified models.")
        return

    g = sns.FacetGrid(
        data,
        col="Model_Title",
        col_order=model_titles,
        height=PLOT_CFG["height"],
        aspect=PLOT_CFG["aspect"],
        sharey=True,
        margin_titles=False
    )

    for ax, title in zip(g.axes.flat, model_titles):
        sub = data[data["Model_Title"] == title]
        for metric, mdf in sub.groupby("Metric", sort=False):
            mdf = mdf.sort_values("step")
            x = mdf["step"].to_numpy()
            y = mdf["Mean"].to_numpy()
            s = mdf["Std"].to_numpy()
            color = COLOR_MAP.get(metric, None)

            if PLOT_CFG["use_dashed"]:
                ax.plot(x, y, label=str(metric), color=color, dashes=LINESTYLE_MAP.get(metric, (6, 3)))
            else:
                ax.plot(x, y, label=str(metric), color=color)

            ax.fill_between(x, y - s, y + s, color=color, alpha=PLOT_CFG["shade_alpha"])

        ax.set_title(title, fontsize=14, weight="bold", style="italic")
        ax.set_xlabel("Training Steps", fontsize=12)
        ax.grid(True, alpha=0.3)

    g.axes.flat[0].set_ylabel("Accuracy (%)", fontsize=12)

    handles, labels = g.axes.flat[-1].get_legend_handles_labels()
    g.axes.flat[-1].legend(handles=handles, labels=labels, loc=PLOT_CFG["legend_loc"], frameon=True)

    out = outfile or PLOT_CFG["outfile"]
    g.fig.savefig(out, dpi=PLOT_CFG["dpi"], bbox_inches="tight", format="pdf")
    print(f"Figure saved as '{out}'")
    plt.show()

# =============================================================================
# 4) Execution
# =============================================================================
if __name__ == "__main__":
    setup_style()
    csv_path = "gpt2train/probe_results_multi.csv"   # Per-seed raw results CSV
    stats_df = load_stats(csv_path)
    print(f"Stats: {stats_df.shape}")

    # Compare noise 0/1/5 models side by side
    models_to_display = [
        'bioS_noise_0_multiple-context_250818-124537',
        'bioS_noise_1_multiple-context_passage-noise_250816-095848',
        # 'bioS_noise_5_multiple-context_passage-noise_250816-125421',
        # 'bioS_noise_10_multiple-context_passage-noise_250816-155036'
    ]
    plot_with_std(stats_df, models_to_display)

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from typing import Dict, List

# =============================================================================
# 0) Configuration block (only modify here)
# =============================================================================
PLOT_CFG = {
    # Per-facet size
    "height": 2.0,        # subplot height
    "aspect": 1.7,        # subplot width/height ratio
    # Line / shading
    "use_dashed": True,
    "linewidth": 2.0,
    "shade_alpha": 0.18,
    # Font / style
    "style": "whitegrid",
    "palette": "deep",
    "dpi": 150,
    # Axis labels
    "xlabel": "Training Steps",
    "ylabel": "Accuracy (%)",
    # Legend (outside figure)
    "legend_out": True,
    "legend_loc": "center left",
    "legend_bbox": (1.02, 0.5),
    "legend_ncol": 1,
    "legend_fontsize": 11,
    "right_pad": 1,    # Right margin (legend space)
    # Output filename
    "outfile": "result_noise01.pdf",
}

# =============================================================================
# 1) Model name -> panel title (noise 0 / 1 only; noise=5 removed)
# =============================================================================
TITLE_MAP: Dict[str, str] = {
    'bioS_noise_0_multiple-context_250818-124537': "without noise",
    'bioS_noise_1_multiple-context_passage-noise_250816-095848': "1% noise",
}

# =============================================================================
# 2) CSV column name -> legend label (LaTeX)
# =============================================================================
METRIC_MAP: Dict[str, str] = {
    'em/param': r'$\mathrm{Acc}_{\mathrm{PKU}}$',
    'em/multi_ood_in_ctx': r'$\mathrm{Acc}_{\mathrm{ICKU}}$',
    'em/pert_ctx_orig': r'$\mathrm{Pref}_{\mathrm{PK}}$',
    'em/pert_ctx_pert': r'$\mathrm{Pref}_{\mathrm{ICK}}$',
}

# =============================================================================
# 3) Color / line style map (by label)
# =============================================================================
COLOR_MAP = {
    r'$\mathrm{Acc}_{\mathrm{PKU}}$': sns.color_palette("deep")[2],
    r'$\mathrm{Acc}_{\mathrm{ICKU}}$': sns.color_palette("deep")[0],
    r'$\mathrm{Pref}_{\mathrm{PK}}$':  sns.color_palette("deep")[3],
    r'$\mathrm{Pref}_{\mathrm{ICK}}$': sns.color_palette("deep")[1],
}
#all different line styles
LINESTYLE_MAP = {
    r'$\mathrm{Acc}_{\mathrm{PKU}}$': (2, 2),
    r'$\mathrm{Acc}_{\mathrm{ICKU}}$': (2, 2),
    r'$\mathrm{Pref}_{\mathrm{PK}}$':  (2, 4),
    r'$\mathrm{Pref}_{\mathrm{ICK}}$': (2, 4),
}

# =============================================================================
# 4) Style setup (control size via FacetGrid height/aspect only)
# =============================================================================
def setup_style() -> None:
    sns.set_theme(
        style=PLOT_CFG["style"],
        palette=PLOT_CFG["palette"],
        rc={
            "figure.dpi": PLOT_CFG["dpi"],
            "axes.titlesize": 14,
            "axes.labelsize": 12,
            "legend.fontsize": PLOT_CFG["legend_fontsize"],
            "lines.linewidth": PLOT_CFG["linewidth"],
        }
    )

# =============================================================================
# 5) Data loading & statistics aggregation (per-seed -> mean/std)
# =============================================================================
def load_stats(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    # Rename only metrics of interest to labels
    rename_cols = {k: v for k, v in METRIC_MAP.items() if k in df.columns}
    df = df.rename(columns=rename_cols)
    metrics = list(rename_cols.values())

    # long-form (keep seed)
    long_seed = pd.melt(
        df,
        id_vars=["model", "step", "seed"],
        value_vars=[c for c in metrics if c in df.columns],
        var_name="Metric",
        value_name="Accuracy"
    )

    # Convert to % if 0~1 scale
    if len(long_seed) and long_seed["Accuracy"].max() <= 1.0:
        long_seed["Accuracy"] *= 100.0

    # Model title mapping
    long_seed["Model_Title"] = long_seed["model"].map(TITLE_MAP).fillna(long_seed["model"])

    # mean/std per (model, step, Metric)
    stats = (
        long_seed
        .groupby(["model", "Model_Title", "step", "Metric"], as_index=False)
        .agg(Mean=("Accuracy", "mean"), Std=("Accuracy", "std"), N=("Accuracy", "count"))
        .sort_values(["Model_Title", "Metric", "step"])
        .reset_index(drop=True)
    )
    return stats

# =============================================================================
# 6) Plotting (mean + std shading, legend outside figure)
# =============================================================================
def plot_with_std(stats_df: pd.DataFrame, models: List[str], outfile: str = None) -> None:
    model_titles = [TITLE_MAP.get(m, m) for m in models]
    data = stats_df[stats_df["Model_Title"].isin(model_titles)].copy()
    if data.empty:
        print("No data found for specified models.")
        return

    g = sns.FacetGrid(
        data,
        col="Model_Title",
        col_order=model_titles,
        height=PLOT_CFG["height"],
        aspect=PLOT_CFG["aspect"],
        sharey=True,
        margin_titles=False
    )

    for ax, title in zip(g.axes.flat, model_titles):
        sub = data[data["Model_Title"] == title]
        for metric, mdf in sub.groupby("Metric", sort=False):
            mdf = mdf.sort_values("step")
            x = mdf["step"].to_numpy()
            y = mdf["Mean"].to_numpy()
            s = mdf["Std"].to_numpy()
            color = COLOR_MAP.get(metric, None)

            if PLOT_CFG["use_dashed"]:
                ax.plot(x, y, label=str(metric), color=color, dashes=LINESTYLE_MAP.get(metric, (6, 3)))
            else:
                ax.plot(x, y, label=str(metric), color=color)

            ax.fill_between(x, y - s, y + s, color=color, alpha=PLOT_CFG["shade_alpha"])

        ax.set_title(title, fontsize=14, weight="bold", style="italic")
        ax.set_xlabel(PLOT_CFG["xlabel"], fontsize=12)
        ax.grid(True, alpha=0.3)

    # y label on the first axis only
    g.axes.flat[0].set_ylabel(PLOT_CFG["ylabel"], fontsize=12)

    # ---- Global legend outside figure ----
    handles_labels = [ax.get_legend_handles_labels() for ax in g.axes.flat]
    handles = sum((hl[0] for hl in handles_labels), [])
    labels  = sum((hl[1] for hl in handles_labels), [])
    # Deduplicate (keep first occurrence)
    uniq = {}
    for h, lab in zip(handles, labels):
        if lab not in uniq:
            uniq[lab] = h

    if PLOT_CFG["legend_out"]:
        g.fig.legend(
            list(uniq.values()), list(uniq.keys()),
            loc=PLOT_CFG["legend_loc"],
            bbox_to_anchor=PLOT_CFG["legend_bbox"],
            ncol=PLOT_CFG["legend_ncol"],
            frameon=True,
            prop={"size": PLOT_CFG["legend_fontsize"]},
        )
        g.fig.subplots_adjust(right=PLOT_CFG["right_pad"])
    
    # Remove per-axis legends if present
    for ax in g.axes.flat:
        leg = ax.get_legend()
        if leg:
            leg.remove()

    out = outfile or PLOT_CFG["outfile"]
    g.fig.savefig(out, dpi=PLOT_CFG["dpi"], bbox_inches="tight", format="pdf")
    print(f"Figure saved as '{out}'")
    plt.show()

# =============================================================================
# 7) Execution
# =============================================================================
if __name__ == "__main__":
    setup_style()
    csv_path = "gpt2train/probe_results_multi.csv"   # Per-seed raw results CSV
    stats_df = load_stats(csv_path)
    print(f"Stats: {stats_df.shape}")

    # Compare noise 0/1 models only (noise=5 removed)
    models_to_display = [
        'bioS_noise_0_multiple-context_250818-124537',                   # No noise
        'bioS_noise_1_multiple-context_passage-noise_250816-095848'      # 1% noise
    ]
    plot_with_std(stats_df, models_to_display)

In [ ]:
# bars_acc_icku_last.py
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

CSV_PATH   = "gpt2train/probe_results_multi.csv"
METRIC_COL = "em/multi_ood_in_ctx"   # ACC_ICKU

MODEL_IDS = [
    'bioS_noise_0_multiple-context_250818-124537',
    'bioS_noise_1_multiple-context_passage-noise_250816-095848',
    'bioS_noise_5_multiple-context_passage-noise_250816-125421',
    'bioS_noise_10_multiple-context_passage-noise_250816-155036',
]
MODEL_LABELS = {
    'bioS_noise_0_multiple-context_250818-124537':  "0%",
    'bioS_noise_1_multiple-context_passage-noise_250816-095848':  "1%",
    'bioS_noise_5_multiple-context_passage-noise_250816-125421':  "5%",
    'bioS_noise_10_multiple-context_passage-noise_250816-155036': "10%",
}
ORDER = ["0%", "1%", "5%", "10%"]

sns.set_theme(style="whitegrid", rc={
    "figure.dpi": 150,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

# Data loading
df = pd.read_csv(CSV_PATH)

# Percent scale correction
if METRIC_COL in df.columns and df[METRIC_COL].max() <= 1.0:
    df[METRIC_COL] *= 100.0

# Extract only the last checkpoint per model (keep seed values)
rows = []
for mid in MODEL_IDS:
    sub = df[df["model"] == mid].copy()
    if sub.empty:
        continue
    last_step = sub["step"].max()
    sub_last  = sub[sub["step"] == last_step][["model", "seed", METRIC_COL]].copy()
    sub_last["label"] = MODEL_LABELS.get(mid, mid)
    rows.append(sub_last)

if rows:
    last_df = pd.concat(rows, ignore_index=True)
    last_df = last_df.dropna(subset=[METRIC_COL])
    last_df["label"] = pd.Categorical(last_df["label"], categories=ORDER, ordered=True)
else:
    last_df = pd.DataFrame(columns=["model", "seed", METRIC_COL, "label"])

# Summary output (optional)
if not last_df.empty:
    summary = (
        last_df.groupby("label", as_index=False)[METRIC_COL]
               .agg(mean="mean", std="std", n="count")
               .sort_values("label")
    )
    print(summary.round(2))
else:
    print("No data available for bar chart.")

# Plotting
fig, ax = plt.subplots(figsize=(2.5, 2.5))
sns.barplot(
    data=last_df,
    x="label", y=METRIC_COL,
    order=ORDER,
    estimator=np.mean,
    errorbar="sd",     # Standard deviation error bar
    capsize=0.2,
    width=0.7,
    ax=ax
)
ax.set_xlabel("Noise")
ax.set_ylabel(r'$\mathrm{Acc}_{\mathrm{ICKU}}$' + "(%)")
# ax.set_title(r'$\mathrm{Acc}_{\mathrm{ICKU}}$' + " at End of Training", weight="bold", style="italic")
ax.grid(True, axis="y", linestyle="--", alpha=0.5)
sns.despine(left=False, bottom=False)

plt.tight_layout()
plt.savefig("bars_acc_icku_last.pdf", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Two-panel histograms with Seaborn (clean, no grid):
  • Left  : zipf + No noise
  • Right : zipf + 1% noise
Within each panel:
  - Blue   : Entropy (param_entropy_nats)   [left y-axis]
  - Orange : Pref_PK (acc_pert_ctx_orig)    [right y-axis]

Bins are shown as *Zipf-rank* groups of 10k (coarsened from 5k by adjacent averaging).
"""

from pathlib import Path
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ===== Settings =====
STEP = 16000
ORIG_BINW = 5000
NEW_BINW  = 10000

RUNS = {
    "zipf_textlist_out_zipf_100_0_250816-104331": "zipf + No noise",
    "zipf_textlist_out_zipf_100_1_250816-151409": "zipf + 1% noise",
}

OUT_PNG = f"zipfrank_bins_dualmetric_{NEW_BINW}_step{STEP}_seaborn_centered.png"
OUT_PDF = f"zipfrank_binsd.pdf"

# ===== Helpers =====
def find_bins_csv(run_name: str, step: int, binw: int) -> Path:
    fname = f"zipf_bins_step{step}_w{binw}_{run_name}_bins.csv"
    p = Path(fname)
    if p.exists():
        return p
    cand = list(Path(".").rglob(fname))
    if cand:
        return cand[0]
    raise FileNotFoundError(f"Cannot find per-bin CSV: {fname}")

def _parse_bound(s: str, side: str) -> int:
    s = str(s).replace("–", "-")
    part = s.split("-")[0 if side=="L" else -1].strip().replace(",", "")
    m = re.findall(r"\d+", part)
    return int(m[0]) if m else 10**12

def parse_left_bound(bin_label: str) -> int:
    return _parse_bound(bin_label, "L")

def parse_right_bound(bin_label: str) -> int:
    return _parse_bound(bin_label, "R")

def fmt_zipfbin_short(bin_label: str) -> str:
    """'0-10000' -> '0-10k' (Zipf-rank 10k bin abbreviated notation)"""
    L = parse_left_bound(bin_label)
    R = parse_right_bound(bin_label)
    return f"{L//1000}–{R//1000}k"

def coarsen_bins_equal_counts(df: pd.DataFrame, orig_w: int, new_w: int) -> pd.DataFrame:
    """new_w must be a multiple of orig_w. (Assumes equal entity count -> adjacent averaging)"""
    assert new_w % orig_w == 0, "new_w must be a multiple of orig_w"
    d = df.copy()
    if "Pref_PK" not in d.columns and "acc_pert_ctx_orig" in d.columns:
        d = d.rename(columns={"acc_pert_ctx_orig": "Pref_PK"})
    d["left"] = d["rank_bin"].apply(parse_left_bound).astype(int)
    d["group_left"] = (d["left"] // new_w) * new_w
    # Zipf-rank 10k bin label
    d["zipf_bin_10k"] = d["group_left"].astype(int).apply(lambda L: f"{L}–{L+new_w}")
    metrics = ["param_entropy_nats", "Pref_PK"]
    grp_cols = ["legend", "group_left", "zipf_bin_10k"]
    out = (
        d.groupby(grp_cols, observed=True)[metrics]
         .mean()
         .reset_index()
         .sort_values(["group_left", "legend"])
    )
    bins_sorted = sorted(out["zipf_bin_10k"].unique(), key=parse_left_bound)
    out["zipf_bin_10k"] = pd.Categorical(out["zipf_bin_10k"], categories=bins_sorted, ordered=True)
    return out

def _offset_bars(ax, dx: float):
    """Shift bar x-coordinates by dx (in data coordinates)."""
    for p in ax.patches:
        p.set_x(p.get_x() + dx)

# ===== Load 5k-bin CSVs & coarsen to Zipf-rank 10k bins =====
dfs = []
for run, label in RUNS.items():
    csv_path = find_bins_csv(run, STEP, ORIG_BINW)
    df = pd.read_csv(csv_path)[["rank_bin", "param_entropy_nats", "acc_pert_ctx_orig"]].copy()
    df["legend"] = label
    dfs.append(df)
all_df_5k = pd.concat(dfs, ignore_index=True)
all_df_10k = coarsen_bins_equal_counts(all_df_5k, ORIG_BINW, NEW_BINW)

# ===== Plot (Seaborn clean, no grid) =====
sns.set_theme(style="ticks", rc={
    "figure.dpi": 170,
    "axes.titlesize": 13,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 13,
    "axes.edgecolor": "0.25",
    "axes.linewidth": 0.8,
})

entropy_color = sns.color_palette("deep")[0]  # blue
prefpk_color  = sns.color_palette("deep")[1]  # orange

fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharex=True)

# Unify Entropy y-range across panels
ent_all = all_df_10k["param_entropy_nats"].to_numpy(float)
ent_all = ent_all[~np.isnan(ent_all)]
ent_min = max(0.0, np.nanmin(ent_all) if ent_all.size else 0.0)
ent_max = (np.nanmax(ent_all) if ent_all.size else 1.0)
marg = 0.06 * (ent_max - ent_min if ent_max > ent_min else 1.0)
ent_ylim = (max(0.0, ent_min - marg), ent_max + marg)

panel_labels = list(RUNS.values())
bar_width = 0.30
offset_dx = bar_width  # Right bar offset -> half for tick centering

for ax, legend_label in zip(axes, panel_labels):
    sub = all_df_10k[all_df_10k["legend"] == legend_label].copy()

    # Left axis: Entropy
    sns.barplot(
        data=sub,
        x="zipf_bin_10k", y="param_entropy_nats",
        ax=ax,
        color=entropy_color,
        width=bar_width,
    )
    ax.set_ylim(*ent_ylim)
    ax.set_ylabel("Entropy (nats)")

    # Right axis: Pref_PK
    ax2 = ax.twinx()
    sns.barplot(
        data=sub,
        x="zipf_bin_10k", y="Pref_PK",
        ax=ax2,
        color=prefpk_color,
        width=bar_width,
        alpha=0.95
    )
    _offset_bars(ax2, dx=offset_dx)
    ax2.set_ylim(0.0, 1.0)
    ax2.set_ylabel("Pref$_{\\mathrm{PK}}$")

    # ==== Place labels at center of two bars, '0-10k' style ====
    bins = list(sub["zipf_bin_10k"].cat.categories)
    centers = np.arange(len(bins)) + offset_dx / 2.0
    ax.set_xticks(centers)
    ax.set_xticklabels([fmt_zipfbin_short(b) for b in bins], rotation=0)
    ax.set_xlabel("Zipf-rank bin (10k width)")

    # Adjust xlim for offset
    ax.set_xlim(-0.55, (len(bins)-0.5) + offset_dx + 0.05)

    # No grid + minimal spines
    ax.grid(False); ax2.grid(False)
    sns.despine(ax=ax,  top=True, right=False, left=False, bottom=False)
    sns.despine(ax=ax2, top=True, right=False, left=False, bottom=False)

    # Panel title (bold italic)
    ax.set_title(legend_label, fontweight='bold', fontstyle='italic')

# Legend (by metric)
legend_handles = [
    Patch(facecolor=entropy_color, label="Entropy"),
    Patch(facecolor=prefpk_color, label="Pref$_{\\mathrm{PK}}$")
]
axes[0].legend(handles=legend_handles, title="", loc="upper left", frameon=False)

plt.tight_layout()
plt.savefig(OUT_PNG, bbox_inches="tight", dpi=300)
plt.savefig(OUT_PDF, bbox_inches="tight")
plt.show()
print(f"Saved: {OUT_PNG}\nSaved: {OUT_PDF}")

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Two-panel histograms with Seaborn (clean, no grid):
  • Left  : zipf + No noise
  • Right : zipf + 1% noise
Within each panel:
  - Orange : Pref_PK (acc_pert_ctx_orig) [left y-axis]
  - Blue   : Entropy (param_entropy_nats) [right y-axis]

Bins are shown as *Zipf-rank* groups of 10k (coarsened from 5k by adjacent averaging).
"""

from pathlib import Path
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ===== Settings =====
STEP = 16000
ORIG_BINW = 5000
NEW_BINW  = 10000

RUNS = {
    "zipf_textlist_out_zipf_100_0_250816-104331": "Zipfian w/o noise",
    "zipf_textlist_out_zipf_100_1_250816-151409": "Zipfian w/ 1% noise",
}

OUT_PNG = f"zipfrank_bins_dualmetric_{NEW_BINW}_step{STEP}_seaborn_centered.png"
OUT_PDF = f"zipfrank_binsd.pdf"

# ===== Helpers =====
def find_bins_csv(run_name: str, step: int, binw: int) -> Path:
    fname = f"zipf_bins_step{step}_w{binw}_{run_name}_bins.csv"
    p = Path(fname)
    if p.exists():
        return p
    cand = list(Path(".").rglob(fname))
    if cand:
        return cand[0]
    raise FileNotFoundError(f"Cannot find per-bin CSV: {fname}")

def _parse_bound(s: str, side: str) -> int:
    s = str(s).replace("–", "-")
    part = s.split("-")[0 if side=="L" else -1].strip().replace(",", "")
    m = re.findall(r"\d+", part)
    return int(m[0]) if m else 10**12

def parse_left_bound(bin_label: str) -> int:
    return _parse_bound(bin_label, "L")

def parse_right_bound(bin_label: str) -> int:
    return _parse_bound(bin_label, "R")

def fmt_zipfbin_short(bin_label: str) -> str:
    """'0-10000' -> '0-10k' (Zipf-rank 10k bin abbreviated notation), but show 0 as 1"""
    L = parse_left_bound(bin_label)
    R = parse_right_bound(bin_label)
    if L == 0:
        return "1-10k"
    return f"{L//1000}k–{R//1000}k"

def coarsen_bins_equal_counts(df: pd.DataFrame, orig_w: int, new_w: int) -> pd.DataFrame:
    """new_w must be a multiple of orig_w. (Assumes equal entity count -> adjacent averaging)"""
    assert new_w % orig_w == 0, "new_w must be a multiple of orig_w"
    d = df.copy()
    if "Pref_PK" not in d.columns and "acc_pert_ctx_orig" in d.columns:
        d = d.rename(columns={"acc_pert_ctx_orig": "Pref_PK"})
    d["left"] = d["rank_bin"].apply(parse_left_bound).astype(int)
    d["group_left"] = (d["left"] // new_w) * new_w
    # Zipf-rank 10k bin label
    d["zipf_bin_10k"] = d["group_left"].astype(int).apply(lambda L: f"{L}–{L+new_w}")
    metrics = ["param_entropy_nats", "Pref_PK"]
    grp_cols = ["legend", "group_left", "zipf_bin_10k"]
    out = (
        d.groupby(grp_cols, observed=True)[metrics]
         .mean()
         .reset_index()
         .sort_values(["group_left", "legend"])
    )
    bins_sorted = sorted(out["zipf_bin_10k"].unique(), key=parse_left_bound)
    out["zipf_bin_10k"] = pd.Categorical(out["zipf_bin_10k"], categories=bins_sorted, ordered=True)
    return out

def _offset_bars(ax, dx: float):
    """Shift bar x-coordinates by dx (in data coordinates)."""
    for p in ax.patches:
        p.set_x(p.get_x() + dx)

# ===== Load 5k-bin CSVs & coarsen to Zipf-rank 10k bins =====
dfs = []
for run, label in RUNS.items():
    csv_path = find_bins_csv(run, STEP, ORIG_BINW)
    df = pd.read_csv(csv_path)[["rank_bin", "param_entropy_nats", "acc_pert_ctx_orig"]].copy()
    df["legend"] = label
    dfs.append(df)
all_df_5k = pd.concat(dfs, ignore_index=True)
all_df_10k = coarsen_bins_equal_counts(all_df_5k, ORIG_BINW, NEW_BINW)

# ===== Plot (Seaborn clean, no grid) =====
sns.set_theme(style="ticks", rc={
    "figure.dpi": 170,
    "axes.titlesize": 13,
    "axes.labelsize": 13,
    "xtick.labelsize": 9,
    "ytick.labelsize": 13,
    "axes.edgecolor": "0.25",
    "axes.linewidth": 0.8,
})


entropy_color = sns.color_palette("deep")[9]  # blue
prefpk_color  = sns.color_palette("deep")[3]  # red

fig, axes = plt.subplots(1, 2, figsize=(10, 2.5), sharex=True)

# Unify Entropy y-range across panels
ent_all = all_df_10k["param_entropy_nats"].to_numpy(float)
ent_all = ent_all[~np.isnan(ent_all)]
ent_min = max(0.0, np.nanmin(ent_all) if ent_all.size else 0.0)
ent_max = (np.nanmax(ent_all) if ent_all.size else 1.0)
marg = 0.06 * (ent_max - ent_min if ent_max > ent_min else 1.0)
ent_ylim = (max(0.0, ent_min - marg), ent_max + marg)

panel_labels = list(RUNS.values())
bar_width = 0.30
offset_dx = bar_width  # Right bar offset -> half for tick centering

for ax, legend_label in zip(axes, panel_labels):
    sub = all_df_10k[all_df_10k["legend"] == legend_label].copy()

    # === Left axis: Pref_PK (orange) ===
    sns.barplot(
        data=sub,
        x="zipf_bin_10k", y="Pref_PK",
        ax=ax,
        color=prefpk_color,
        width=bar_width,
    )
    ax.set_ylim(0.0, 1.0)
    ax.set_ylabel("Pref$_{\\mathrm{PK}}$", color=prefpk_color)
    ax.tick_params(axis="y", colors=prefpk_color)
    ax.spines["left"].set_color(prefpk_color)

    # === Right axis: Entropy (blue) ===
    ax2 = ax.twinx()
    sns.barplot(
        data=sub,
        x="zipf_bin_10k", y="param_entropy_nats",
        ax=ax2,
        color=entropy_color,
        width=bar_width,
        alpha=0.95
    )
    _offset_bars(ax2, dx=offset_dx)
    ax2.set_ylim(*ent_ylim)
    ax2.set_ylabel("Entropy (nats)", color=entropy_color)
    ax2.tick_params(axis="y", colors=entropy_color)
    ax2.spines["right"].set_color(entropy_color)

    # ==== Place labels at center of two bars, '0-10k' style ====
    bins = list(sub["zipf_bin_10k"].cat.categories)
    centers = np.arange(len(bins)) + offset_dx / 2.0
    ax.set_xticks(centers)
    ax.set_xticklabels([fmt_zipfbin_short(b) for b in bins], rotation=0)
    ax.set_xlabel("Zipfian rank(10k width)")

    # Adjust xlim for offset
    ax.set_xlim(-0.55, (len(bins)-0.5) + offset_dx + 0.05)

    # No grid + minimal spines
    ax.grid(False); ax2.grid(False)
    sns.despine(ax=ax,  top=True, right=False, left=False, bottom=False)
    sns.despine(ax=ax2, top=True, right=False, left=False, bottom=False)

    # Panel title (bold italic)
    ax.set_title(legend_label, fontweight='bold', fontstyle='italic')

# Legend (by metric)
legend_handles = [
    Patch(facecolor=prefpk_color, label="Pref$_{\\mathrm{PK}}$"),
    Patch(facecolor=entropy_color, label="Entropy"),
]
axes[0].legend(handles=legend_handles, title="", loc="upper left", frameon=False)

plt.tight_layout()
plt.savefig(OUT_PNG, bbox_inches="tight", dpi=300)
plt.savefig(OUT_PDF, bbox_inches="tight")
plt.show()
print(f"Saved: {OUT_PNG}\nSaved: {OUT_PDF}")

In [ ]:
# === Jupyter one-cell: legend outside, no gap plot ===
# Updated for 6 groups: top/bottom 1%, 5%, 10%

import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Dict

# =============================================================================
# 0) Configuration block
# =============================================================================
PLOT_CFG = {
    # Facet size
    "height": 2.0,
    "aspect": 1.5,

    # Line style
    "use_dashed": True,
    "linewidth": 2.0,

    # Font / style
    "style": "whitegrid",
    "palette": "deep",
    "dpi": 150,

    # Axis labels
    "xlabel": "Training Steps",
    "ylabel": "Accuracy (%)",

    # Legend (outside figure)
    "legend_out": True,
    "legend_loc": "center left",
    "legend_bbox": (1.02, 0.5),
    "legend_ncol": 1,
    "legend_fontsize": 11,
    "right_pad": 0.82,

    # Save
    "save": True,
    "save_dir": "./",
    "save_formats": ("png","pdf"),
    "save_dpi": 300,
    "outfile_facet": "zipf_em_topbottom",
}

# Input / filter
CSV_PATH     = "gpt2train/probe_zipf_em_topbottom_1_5_10p_summary.csv"
MODEL_FILTER = None     # e.g.: "zipf"
STEP_MIN     = None     # e.g.: 1000
STEP_MAX     = None     # e.g.: 32000
YLIM         = (0, 100) # None for auto

# =============================================================================
# 1) Label / group definitions (6 groups: top/bottom 1%, 5%, 10%)
# =============================================================================
LABELS_LATEX: Dict[str, str] = {
    "param":          r"$\mathrm{Acc}_{\mathrm{PK}}$",
    "pert_ctx_orig":  r"$\mathrm{Pref}_{\mathrm{PK}}$",
    "pert_ctx_pert":  r"$\mathrm{Pref}_{\mathrm{ICK}}$",
}
# Output only top5, bottom5
GROUP_NAME_MAP = {
    "top5p": "Top 5%",
    "bottom5p": "Bottom 5%",
}
GROUP_ORDER = ["Top 5%", "Bottom 5%"]
METRIC_MODES = ["param", "pert_ctx_orig", "pert_ctx_pert"]
GROUPS_RAW = ["top5p", "bottom5p"]

# =============================================================================
# 2) Style setup
# =============================================================================
def setup_style() -> None:
    mpl.rcParams["figure.dpi"] = PLOT_CFG["dpi"]
    plt.rcParams["pdf.fonttype"] = 42
    plt.rcParams["ps.fonttype"]  = 42
    sns.set_theme(
        style=PLOT_CFG["style"],
        palette=PLOT_CFG["palette"],
        rc={
            "axes.titlesize": 14,
            "axes.labelsize": 12,
            "legend.fontsize": PLOT_CFG["legend_fontsize"],
            "lines.linewidth": PLOT_CFG["linewidth"],
        }
    )

def _color_cycle():
    pal = sns.color_palette(PLOT_CFG["palette"])
    # Fixed 3-color mapping for readability (preserving deep palette feel)
    return [pal[2], pal[3], pal[1]]

COLOR_MAP = dict(zip(LABELS_LATEX.values(), _color_cycle()))
LINESTYLE_MAP = {lab: (2,2) for lab in LABELS_LATEX.values()}

# =============================================================================
# 3) Utilities
# =============================================================================
def ensure_cols(df: pd.DataFrame):
    needed = ["model","step"]
    for base in METRIC_MODES:
        for g in GROUPS_RAW:
            # Check for _mean suffix columns
            needed.append(f"{base}_{g}_mean")
    miss = [c for c in needed if c not in df.columns]
    if miss:
        raise ValueError(f"Required columns missing from CSV: {miss}\n=> Actual columns: {list(df.columns)}")

def sanitize(name: str) -> str:
    return re.sub(r"[^-\w.]+", "_", str(name))

def filter_df(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["step"] = pd.to_numeric(out["step"], errors="coerce")
    out = out.dropna(subset=["step"])
    out["step"] = out["step"].astype(int)
    if MODEL_FILTER:
        out = out[out["model"].astype(str).str.contains(MODEL_FILTER, na=False)]
    if STEP_MIN is not None:
        out = out[out["step"] >= STEP_MIN]
    if STEP_MAX is not None:
        out = out[out["step"] <= STEP_MAX]
    return out

def to_long(df: pd.DataFrame) -> pd.DataFrame:
    # Use only _mean columns (exclude std)
    value_cols = [c for c in df.columns if c.endswith("_mean") and any(c.startswith(b+"_") for b in METRIC_MODES)]
    long_df = df.melt(
        id_vars=[c for c in ["model","step"] if c in df.columns],
        value_vars=value_cols,
        var_name="Key",
        value_name="Value"
    )
    # Key e.g.: "param_top1p_mean" -> Base="param", Group="top1p"
    # Remove _mean suffix then rsplit
    long_df["Key_clean"] = long_df["Key"].str.replace("_mean$", "", regex=True)
    tmp = long_df["Key_clean"].str.rsplit("_", n=1, expand=True)
    long_df["Base"]   = tmp[0]
    long_df["Group"]  = tmp[1].map(GROUP_NAME_MAP)
    long_df["Metric"] = long_df["Base"].map(LABELS_LATEX)
    long_df = long_df.dropna(subset=["Group","Metric"])
    long_df = long_df.sort_values(["model","Group","Metric","step"])
    return long_df

# =============================================================================
# 4) Plot (legend outside figure) - 2x3 layout (Top row: 1%,5%,10% / Bottom row: 1%,5%,10%)
# =============================================================================
def facet_lines_mean(stats_df: pd.DataFrame, model_name: str) -> plt.Figure:
    groups_present = [g for g in GROUP_ORDER if g in stats_df["Group"].unique()]
    
    # Add row column for Top/Bottom separation
    stats_df = stats_df.copy()
    stats_df["Position"] = stats_df["Group"].apply(
        lambda x: "Top" if x.startswith("Top") else "Bottom"
    )
    stats_df["Percentage"] = stats_df["Group"].apply(
        lambda x: x.split()[-1]  # "1%", "5%", "10%"
    )
    
    # 2x3 FacetGrid: row=Position, col=Percentage
    g = sns.FacetGrid(
        stats_df,
        row="Position",
        col="Percentage",
        row_order=["Top", "Bottom"],
        col_order=["1%", "5%", "10%"],
        height=PLOT_CFG["height"],
        aspect=PLOT_CFG["aspect"],
        sharey=True,
        margin_titles=False,
    )
    
    for ax in g.axes.flat:
        ax.set_visible(False)
    
    for (pos, pct), ax in g.axes_dict.items():
        grp = f"{pos} {pct}"
        if grp not in groups_present:
            continue
        ax.set_visible(True)
        sub = stats_df[stats_df["Group"] == grp]
        for metric, mdf in sub.groupby("Metric", sort=False):
            mdf = mdf.sort_values("step")
            color = COLOR_MAP.get(metric)
            (line,) = ax.plot(mdf["step"], mdf["Value"], label=str(metric), color=color)
            if PLOT_CFG["use_dashed"]:
                line.set_dashes(LINESTYLE_MAP.get(metric, (6,3)))
        ax.set_title(grp, fontsize=13, weight="bold", style="italic")
        if YLIM:
            ax.set_ylim(*YLIM)
        ax.grid(True, alpha=0.3)

    # x, y label setup
    for ax in g.axes[-1, :]:
        ax.set_xlabel(PLOT_CFG["xlabel"], fontsize=12)
    for ax in g.axes[:, 0]:
        ax.set_ylabel(PLOT_CFG["ylabel"], fontsize=12)

    # ---- Global legend outside figure ----
    handles_labels = [ax.get_legend_handles_labels() for ax in g.axes.flat if ax.get_visible()]
    handles = sum((hl[0] for hl in handles_labels), [])
    labels  = sum((hl[1] for hl in handles_labels), [])
    uniq = {}
    for h, lab in zip(handles, labels):
        if lab not in uniq:
            uniq[lab] = h
    g.fig.legend(
        list(uniq.values()), list(uniq.keys()),
        loc=PLOT_CFG["legend_loc"],
        bbox_to_anchor=PLOT_CFG["legend_bbox"],
        ncol=PLOT_CFG["legend_ncol"],
        frameon=True,
        prop={"size": PLOT_CFG["legend_fontsize"]},
    )
    g.fig.subplots_adjust(right=PLOT_CFG["right_pad"])
    g.fig.tight_layout()
    return g.fig

def maybe_save(fig: plt.Figure, base: str):
    if not PLOT_CFG["save"]:
        return
    outdir = Path(PLOT_CFG["save_dir"]); outdir.mkdir(parents=True, exist_ok=True)
    base = sanitize(base)
    for fmt in PLOT_CFG["save_formats"]:
        fig.savefig(outdir / f"{base}.{fmt}", dpi=PLOT_CFG["save_dpi"], bbox_inches="tight")

# =============================================================================
# 5) Execution (no difference plot output)
# =============================================================================
setup_style()
df = pd.read_csv(CSV_PATH)
ensure_cols(df)
df = filter_df(df)
if df.empty:
    raise SystemExit("No data matching the conditions. Check CSV_PATH / MODEL_FILTER / STEP_MIN/MAX.")

models = sorted(df["model"].unique())
print(f"{len(models)} model(s): {models}")

for model in models:
    df_m = df[df["model"] == model].copy()
    long_m = to_long(df_m)

    # Top/Bottom 2-facet (mean lines only, legend outside figure)
    fig = facet_lines_mean(long_m, model_name=model)
    plt.show()
    maybe_save(fig, f"{PLOT_CFG['outfile_facet']}__{model}")

print("Done.")

In [ ]:
# Output only top5, bottom5
# === Jupyter one-cell: legend outside, no gap plot ===
# Updated for 6 groups: top/bottom 1%, 5%, 10%

import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Dict

# =============================================================================
# 0) Configuration block (for 1-column figure: horizontal layout)
# =============================================================================
PLOT_CFG = {
    # Facet size (for horizontal layout - 1 column figure)
    "height": 1.8,
    "aspect": 1.3,

    # Line style
    "use_dashed": True,
    "linewidth": 2.0,

    # Font / style
    "style": "whitegrid",
    "palette": "deep",
    "dpi": 150,

    # Axis labels
    "xlabel": "Training Steps",
    "ylabel": "Accuracy (%)",

    # Legend (below figure)
    "legend_out": True,
    "legend_loc": "upper center",
    "legend_bbox": (0.5, -0.18),
    "legend_ncol": 3,
    "legend_fontsize": 9,

    # Save
    "save": True,
    "save_dir": "./",
    "save_formats": ("png","pdf"),
    "save_dpi": 300,
    "outfile_facet": "zipf_em_topbottom_1col",
}

# Input / filter
CSV_PATH     = "gpt2train/probe_zipf_em_topbottom_1_5_10p_summary.csv"
MODEL_FILTER = None     # e.g.: "zipf"
STEP_MIN     = None     # e.g.: 1000
STEP_MAX     = None     # e.g.: 32000
YLIM         = (0, 100) # None for auto

# =============================================================================
# 1) Label / group definitions (6 groups: top/bottom 1%, 5%, 10%)
# =============================================================================
LABELS_LATEX: Dict[str, str] = {
    "param":          r"$\mathrm{Acc}_{\mathrm{PK}}$",
    "pert_ctx_orig":  r"$\mathrm{Pref}_{\mathrm{PK}}$",
    "pert_ctx_pert":  r"$\mathrm{Pref}_{\mathrm{ICK}}$",
}
# Output only top5, bottom5
GROUP_NAME_MAP = {
    "top5p": "Top 5%",
    "bottom5p": "Bottom 5%",
}
GROUP_ORDER = ["Top 5%", "Bottom 5%"]
METRIC_MODES = ["param", "pert_ctx_orig", "pert_ctx_pert"]
GROUPS_RAW = ["top5p", "bottom5p"]

# =============================================================================
# 2) Style setup
# =============================================================================
def setup_style() -> None:
    mpl.rcParams["figure.dpi"] = PLOT_CFG["dpi"]
    plt.rcParams["pdf.fonttype"] = 42
    plt.rcParams["ps.fonttype"]  = 42
    sns.set_theme(
        style=PLOT_CFG["style"],
        palette=PLOT_CFG["palette"],
        rc={
            "axes.titlesize": 14,
            "axes.labelsize": 12,
            "legend.fontsize": PLOT_CFG["legend_fontsize"],
            "lines.linewidth": PLOT_CFG["linewidth"],
        }
    )

def _color_cycle():
    pal = sns.color_palette(PLOT_CFG["palette"])
    # Fixed 3-color mapping for readability (preserving deep palette feel)
    return [pal[2], pal[3], pal[1]]

COLOR_MAP = dict(zip(LABELS_LATEX.values(), _color_cycle()))
LINESTYLE_MAP = {lab: (2,2) for lab in LABELS_LATEX.values()}

# =============================================================================
# 3) Utilities
# =============================================================================
def ensure_cols(df: pd.DataFrame):
    needed = ["model","step"]
    for base in METRIC_MODES:
        for g in GROUPS_RAW:
            # Check for _mean suffix columns
            needed.append(f"{base}_{g}_mean")
    miss = [c for c in needed if c not in df.columns]
    if miss:
        raise ValueError(f"Required columns missing from CSV: {miss}\n=> Actual columns: {list(df.columns)}")

def sanitize(name: str) -> str:
    return re.sub(r"[^-\w.]+", "_", str(name))

def filter_df(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["step"] = pd.to_numeric(out["step"], errors="coerce")
    out = out.dropna(subset=["step"])
    out["step"] = out["step"].astype(int)
    if MODEL_FILTER:
        out = out[out["model"].astype(str).str.contains(MODEL_FILTER, na=False)]
    if STEP_MIN is not None:
        out = out[out["step"] >= STEP_MIN]
    if STEP_MAX is not None:
        out = out[out["step"] <= STEP_MAX]
    return out

def to_long(df: pd.DataFrame) -> pd.DataFrame:
    # Use only _mean columns (exclude std)
    value_cols = [c for c in df.columns if c.endswith("_mean") and any(c.startswith(b+"_") for b in METRIC_MODES)]
    long_df = df.melt(
        id_vars=[c for c in ["model","step"] if c in df.columns],
        value_vars=value_cols,
        var_name="Key",
        value_name="Value"
    )
    # Key e.g.: "param_top1p_mean" -> Base="param", Group="top1p"
    # Remove _mean suffix then rsplit
    long_df["Key_clean"] = long_df["Key"].str.replace("_mean$", "", regex=True)
    tmp = long_df["Key_clean"].str.rsplit("_", n=1, expand=True)
    long_df["Base"]   = tmp[0]
    long_df["Group"]  = tmp[1].map(GROUP_NAME_MAP)
    long_df["Metric"] = long_df["Base"].map(LABELS_LATEX)
    long_df = long_df.dropna(subset=["Group","Metric"])
    long_df = long_df.sort_values(["model","Group","Metric","step"])
    return long_df

# =============================================================================
# 4) Plot (legend below figure) - Top 5% / Bottom 5% horizontal layout (1-column)
# =============================================================================
def facet_lines_mean(stats_df: pd.DataFrame, model_name: str) -> plt.Figure:
    groups_present = [g for g in GROUP_ORDER if g in stats_df["Group"].unique()]
    
    # Horizontal layout: col="Group"
    g = sns.FacetGrid(
        stats_df,
        col="Group",
        col_order=groups_present,
        height=PLOT_CFG["height"],
        aspect=PLOT_CFG["aspect"],
        sharey=True,
        margin_titles=False,
    )
    
    for ax, grp in zip(g.axes.flat, groups_present):
        sub = stats_df[stats_df["Group"] == grp]
        for metric, mdf in sub.groupby("Metric", sort=False):
            mdf = mdf.sort_values("step")
            color = COLOR_MAP.get(metric)
            (line,) = ax.plot(mdf["step"], mdf["Value"], label=str(metric), color=color)
            if PLOT_CFG["use_dashed"]:
                line.set_dashes(LINESTYLE_MAP.get(metric, (6,3)))
        ax.set_title(grp, fontsize=12, weight="bold", style="italic")
        ax.set_xlabel(PLOT_CFG["xlabel"], fontsize=10)
        if YLIM:
            ax.set_ylim(*YLIM)
        ax.grid(True, alpha=0.3)

    g.axes.flat[0].set_ylabel(PLOT_CFG["ylabel"], fontsize=10)

    # ---- Global legend below figure ----
    handles_labels = [ax.get_legend_handles_labels() for ax in g.axes.flat]
    handles = sum((hl[0] for hl in handles_labels), [])
    labels  = sum((hl[1] for hl in handles_labels), [])
    uniq = {}
    for h, lab in zip(handles, labels):
        if lab not in uniq:
            uniq[lab] = h
    g.fig.legend(
        list(uniq.values()), list(uniq.keys()),
        loc=PLOT_CFG["legend_loc"],
        bbox_to_anchor=PLOT_CFG["legend_bbox"],
        ncol=PLOT_CFG["legend_ncol"],
        frameon=True,
        prop={"size": PLOT_CFG["legend_fontsize"]},
    )
    g.fig.tight_layout()
    plt.subplots_adjust(bottom=0.22)  # Reserve space for legend
    return g.fig

def maybe_save(fig: plt.Figure, base: str):
    if not PLOT_CFG["save"]:
        return
    outdir = Path(PLOT_CFG["save_dir"]); outdir.mkdir(parents=True, exist_ok=True)
    base = sanitize(base)
    for fmt in PLOT_CFG["save_formats"]:
        fig.savefig(outdir / f"{base}.{fmt}", dpi=PLOT_CFG["save_dpi"], bbox_inches="tight")

# =============================================================================
# 5) Execution (no difference plot output)
# =============================================================================
setup_style()
df = pd.read_csv(CSV_PATH)
ensure_cols(df)
df = filter_df(df)
if df.empty:
    raise SystemExit("No data matching the conditions. Check CSV_PATH / MODEL_FILTER / STEP_MIN/MAX.")

models = sorted(df["model"].unique())
print(f"{len(models)} model(s): {models}")

for model in models:
    df_m = df[df["model"] == model].copy()
    long_m = to_long(df_m)

    # Top/Bottom 2-facet (mean lines only, legend outside figure)
    fig = facet_lines_mean(long_m, model_name=model)
    plt.show()
    maybe_save(fig, f"{PLOT_CFG['outfile_facet']}__{model}")

print("Done.")

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from typing import Dict, List

# =============================================================================
# 0) Configuration block (only modify here)
# =============================================================================
PLOT_CFG = {
    # Per-facet size
    "height": 2.0,        # subplot height
    "aspect": 1.7,        # subplot width/height ratio
    # Line / shading
    "use_dashed": True,
    "linewidth": 2.0,
    "shade_alpha": 0.18,
    # Font / style
    "style": "whitegrid",
    "palette": "deep",
    "dpi": 150,
    # Axis labels
    "xlabel": "Training Steps",
    "ylabel": "Accuracy (%)",
    # Legend (outside figure)
    "legend_out": True,
    "legend_loc": "center left",
    "legend_bbox": (1.02, 0.5),
    "legend_ncol": 1,
    "legend_fontsize": 11,
    "right_pad": 1,    # Right margin (legend space)
    # Output filename
    "outfile": "result_noise01.pdf",
}

# =============================================================================
# 1) Model name -> panel title (noise 0 / 1 only; noise=5 removed)
# =============================================================================
TITLE_MAP: Dict[str, str] = {
    'bioS_noise_1_multiple-context_passage-noise_250816-095848': "1% noise",
    'bioS_noise_5_multiple-context_passage-noise_250816-125421': "5% noise",
    'bioS_noise_10_multiple-context_passage-noise_250816-155036': "10% noise",
    'bioS_noise_0_multiple-context_250818-124537': "50k Entities",
    'bioS_noise_0_100k_multiple-context_250901-083541': "100k Entities",
    'bioS_noise_0_200k_multiple-context_250901-052022': "200k Entities",
    'zip_text_aug_zipf_50_1_250901-114016': "alpha = 0.5",
    'zipf_textlist_out_zipf_100_1_250816-151409': "alpha = 1.0",
    'zip_text_aug_zipf_200_1_250901-160857': "alpha = 2.0",
}

# =============================================================================
# 2) CSV column name -> legend label (LaTeX)
# =============================================================================
METRIC_MAP: Dict[str, str] = {
    'em/param': r'$\mathrm{Acc}_{\mathrm{PKU}}$',
    'em/multi_ood_in_ctx': r'$\mathrm{Acc}_{\mathrm{ICKU}}$',
    'em/pert_ctx_orig': r'$\mathrm{Pref}_{\mathrm{PK}}$',
    'em/pert_ctx_pert': r'$\mathrm{Pref}_{\mathrm{ICK}}$',
}

# =============================================================================
# 3) Color / line style map (by label)
# =============================================================================
COLOR_MAP = {
    r'$\mathrm{Acc}_{\mathrm{PKU}}$': sns.color_palette("deep")[2],
    r'$\mathrm{Acc}_{\mathrm{ICKU}}$': sns.color_palette("deep")[0],
    r'$\mathrm{Pref}_{\mathrm{PK}}$':  sns.color_palette("deep")[3],
    r'$\mathrm{Pref}_{\mathrm{ICK}}$': sns.color_palette("deep")[1],
}
LINESTYLE_MAP = {
    r'$\mathrm{Acc}_{\mathrm{PKU}}$': (2, 2),
    r'$\mathrm{Acc}_{\mathrm{ICKU}}$': (2, 2),
    r'$\mathrm{Pref}_{\mathrm{PK}}$':  (2, 2),
    r'$\mathrm{Pref}_{\mathrm{ICK}}$': (2, 2),
}

# =============================================================================
# 4) Style setup (control size via FacetGrid height/aspect only)
# =============================================================================
def setup_style() -> None:
    sns.set_theme(
        style=PLOT_CFG["style"],
        palette=PLOT_CFG["palette"],
        rc={
            "figure.dpi": PLOT_CFG["dpi"],
            "axes.titlesize": 14,
            "axes.labelsize": 12,
            "legend.fontsize": PLOT_CFG["legend_fontsize"],
            "lines.linewidth": PLOT_CFG["linewidth"],
        }
    )

# =============================================================================
# 5) Data loading & statistics aggregation (per-seed -> mean/std)
# =============================================================================
def load_stats(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    # Rename only metrics of interest to labels
    rename_cols = {k: v for k, v in METRIC_MAP.items() if k in df.columns}
    df = df.rename(columns=rename_cols)
    metrics = list(rename_cols.values())

    # long-form (keep seed)
    long_seed = pd.melt(
        df,
        id_vars=["model", "step", "seed"],
        value_vars=[c for c in metrics if c in df.columns],
        var_name="Metric",
        value_name="Accuracy"
    )

    # Convert to % if 0~1 scale
    if len(long_seed) and long_seed["Accuracy"].max() <= 1.0:
        long_seed["Accuracy"] *= 100.0

    # Model title mapping
    long_seed["Model_Title"] = long_seed["model"].map(TITLE_MAP).fillna(long_seed["model"])

    # mean/std per (model, step, Metric)
    stats = (
        long_seed
        .groupby(["model", "Model_Title", "step", "Metric"], as_index=False)
        .agg(Mean=("Accuracy", "mean"), Std=("Accuracy", "std"), N=("Accuracy", "count"))
        .sort_values(["Model_Title", "Metric", "step"])
        .reset_index(drop=True)
    )
    return stats

# =============================================================================
# 6) Plotting (mean + std shading, legend outside figure)
# =============================================================================
def plot_with_std(stats_df: pd.DataFrame, models: List[str], outfile: str = None) -> None:
    model_titles = [TITLE_MAP.get(m, m) for m in models]
    data = stats_df[stats_df["Model_Title"].isin(model_titles)].copy()
    if data.empty:
        print("No data found for specified models.")
        return

    g = sns.FacetGrid(
        data,
        col="Model_Title",
        col_order=model_titles,
        height=PLOT_CFG["height"],
        aspect=PLOT_CFG["aspect"],
        sharey=True,
        margin_titles=False
    )

    for ax, title in zip(g.axes.flat, model_titles):
        sub = data[data["Model_Title"] == title]
        for metric, mdf in sub.groupby("Metric", sort=False):
            mdf = mdf.sort_values("step")
            x = mdf["step"].to_numpy()
            y = mdf["Mean"].to_numpy()
            s = mdf["Std"].to_numpy()
            color = COLOR_MAP.get(metric, None)

            if PLOT_CFG["use_dashed"]:
                ax.plot(x, y, label=str(metric), color=color, dashes=LINESTYLE_MAP.get(metric, (6, 3)))
            else:
                ax.plot(x, y, label=str(metric), color=color)

            ax.fill_between(x, y - s, y + s, color=color, alpha=PLOT_CFG["shade_alpha"])

        ax.set_title(title, fontsize=14, weight="bold", style="italic")
        ax.set_xlabel(PLOT_CFG["xlabel"], fontsize=12)
        ax.grid(True, alpha=0.3)

    # y label on the first axis only
    g.axes.flat[0].set_ylabel(PLOT_CFG["ylabel"], fontsize=12)

    # ---- Global legend outside figure ----
    handles_labels = [ax.get_legend_handles_labels() for ax in g.axes.flat]
    handles = sum((hl[0] for hl in handles_labels), [])
    labels  = sum((hl[1] for hl in handles_labels), [])
    # Deduplicate (keep first occurrence)
    uniq = {}
    for h, lab in zip(handles, labels):
        if lab not in uniq:
            uniq[lab] = h

    if PLOT_CFG["legend_out"]:
        g.fig.legend(
            list(uniq.values()), list(uniq.keys()),
            loc=PLOT_CFG["legend_loc"],
            bbox_to_anchor=PLOT_CFG["legend_bbox"],
            ncol=PLOT_CFG["legend_ncol"],
            frameon=True,
            prop={"size": PLOT_CFG["legend_fontsize"]},
        )
        g.fig.subplots_adjust(right=PLOT_CFG["right_pad"])
    
    # Remove per-axis legends if present
    for ax in g.axes.flat:
        leg = ax.get_legend()
        if leg:
            leg.remove()

    out = outfile or PLOT_CFG["outfile"]
    g.fig.savefig(out, dpi=PLOT_CFG["dpi"], bbox_inches="tight", format="pdf")
    print(f"Figure saved as '{out}'")
    plt.show()

# =============================================================================
# 7) Execution
# =============================================================================
if __name__ == "__main__":
    setup_style()
    csv_path = "gpt2train/probe_results_multi.csv"   # Per-seed raw results CSV
    stats_df = load_stats(csv_path)
    print(f"Stats: {stats_df.shape}")

    # Compare noise 0/1 models only (noise=5 removed)
    models_to_display = [
        'bioS_noise_0_multiple-context_250818-124537',                   # No noise
        'bioS_noise_1_multiple-context_passage-noise_250816-095848'      # 1% noise
    ]
    plot_with_std(stats_df, models_to_display)

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from typing import Dict, List

# =============================================================================
# 0) Configuration block (only modify here)
# =============================================================================
PLOT_CFG = {
    "height": 2.0,        # Per-facet height
    "aspect": 1.7,        # Facet width/height ratio
    "use_dashed": True,
    "linewidth": 2.0,
    "shade_alpha": 0.18,
    "style": "whitegrid",
    "palette": "deep",
    "dpi": 150,
    "xlabel": "Training Steps",
    "ylabel": "Accuracy (%)",
    # Legend (outside figure)
    "legend_out": True,
    "legend_loc": "center left",
    "legend_bbox": (1.0, 0.5),
    "legend_ncol": 1,
    "legend_fontsize": 11,
    "right_pad": 1,  # Right margin (legend space); covered by bbox_inches='tight'
    # Default output filename (recommended to specify individually per call)
    "outfile": "result_noise01.pdf",
}

# =============================================================================
# 1) Model name -> panel title
# =============================================================================
TITLE_MAP: Dict[str, str] = {
    # Noise
    'bioS_noise_1_multiple-context_passage-noise_250816-095848': "1% noise",
    'bioS_noise_5_multiple-context_passage-noise_250816-125421': "5% noise",
    'bioS_noise_10_multiple-context_passage-noise_250816-155036': "10% noise",
    # Entities
    'bioS_noise_0_multiple-context_250818-124537': "50k Entities",
    'bioS_noise_0_100k_multiple-context_250901-083541': "100k Entities",
    'bioS_noise_0_200k_multiple-context_250901-052022': "200k Entities",
    # Skewness (Zipf alpha)
    'zip_text_aug_zipf_50_1_250901-114016': "alpha = 0.5",
    'zipf_textlist_out_zipf_100_1_250816-151409': "alpha = 1.0",
    'zip_text_aug_zipf_200_1_250901-160857': "alpha = 2.0",
}

# =============================================================================
# 2) CSV column name -> legend label (LaTeX)
# =============================================================================
METRIC_MAP: Dict[str, str] = {
    'em/param': r'$\mathrm{Acc}_{\mathrm{PKU}}$',
    'em/multi_ood_in_ctx': r'$\mathrm{Acc}_{\mathrm{ICKU}}$',
    'em/pert_ctx_orig': r'$\mathrm{Pref}_{\mathrm{PK}}$',
    'em/pert_ctx_pert': r'$\mathrm{Pref}_{\mathrm{ICK}}$',
}

# =============================================================================
# 3) Color / line style map (by label)
# =============================================================================
COLOR_MAP = {
    r'$\mathrm{Acc}_{\mathrm{PKU}}$': sns.color_palette("deep")[2],
    r'$\mathrm{Acc}_{\mathrm{ICKU}}$': sns.color_palette("deep")[0],
    r'$\mathrm{Pref}_{\mathrm{PK}}$':  sns.color_palette("deep")[3],
    r'$\mathrm{Pref}_{\mathrm{ICK}}$': sns.color_palette("deep")[1],
}
LINESTYLE_MAP = {
    r'$\mathrm{Acc}_{\mathrm{PKU}}$': (2, 2),
    r'$\mathrm{Acc}_{\mathrm{ICKU}}$': (2, 2),
    r'$\mathrm{Pref}_{\mathrm{PK}}$':  (1, 2),
    r'$\mathrm{Pref}_{\mathrm{ICK}}$': (1, 2),
}

# =============================================================================
# 4) Style setup
# =============================================================================
def setup_style() -> None:
    sns.set_theme(
        style=PLOT_CFG["style"],
        palette=PLOT_CFG["palette"],
        rc={
            "figure.dpi": PLOT_CFG["dpi"],
            "axes.titlesize": 14,
            "axes.labelsize": 12,
            "legend.fontsize": PLOT_CFG["legend_fontsize"],
            "lines.linewidth": PLOT_CFG["linewidth"],
        }
    )

# =============================================================================
# 5) Data loading & statistics aggregation (per-seed -> mean/std)
# =============================================================================
def load_stats(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    # Rename only metrics of interest to labels
    rename_cols = {k: v for k, v in METRIC_MAP.items() if k in df.columns}
    df = df.rename(columns=rename_cols)
    metrics = list(rename_cols.values())

    # long-form (keep seed)
    long_seed = pd.melt(
        df,
        id_vars=["model", "step", "seed"],
        value_vars=[c for c in metrics if c in df.columns],
        var_name="Metric",
        value_name="Accuracy"
    )

    # Convert to % if 0~1 scale
    if len(long_seed) and long_seed["Accuracy"].max() <= 1.0:
        long_seed["Accuracy"] *= 100.0

    # Model title mapping
    long_seed["Model_Title"] = long_seed["model"].map(TITLE_MAP).fillna(long_seed["model"])

    # mean/std per (model, step, Metric)
    stats = (
        long_seed
        .groupby(["model", "Model_Title", "step", "Metric"], as_index=False)
        .agg(Mean=("Accuracy", "mean"), Std=("Accuracy", "std"), N=("Accuracy", "count"))
        .sort_values(["Model_Title", "Metric", "step"])
        .reset_index(drop=True)
    )
    return stats

# =============================================================================
# 6) Plotting (mean + std shading, legend outside figure)
# =============================================================================
def plot_with_std(stats_df: pd.DataFrame, models: List[str], outfile: str = None) -> None:
    model_titles = [TITLE_MAP.get(m, m) for m in models]
    data = stats_df[stats_df["Model_Title"].isin(model_titles)].copy()
    if data.empty:
        print("No data found for specified models.")
        return

    g = sns.FacetGrid(
        data,
        col="Model_Title",
        col_order=model_titles,
        height=PLOT_CFG["height"],
        aspect=PLOT_CFG["aspect"],
        sharey=True,
        margin_titles=False
    )

    for ax, title in zip(g.axes.flat, model_titles):
        sub = data[data["Model_Title"] == title]
        for metric, mdf in sub.groupby("Metric", sort=False):
            mdf = mdf.sort_values("step")
            x = mdf["step"].to_numpy()
            y = mdf["Mean"].to_numpy()
            s = mdf["Std"].to_numpy()
            color = COLOR_MAP.get(metric, None)

            if PLOT_CFG["use_dashed"]:
                ax.plot(x, y, label=str(metric), color=color, dashes=LINESTYLE_MAP.get(metric, (6, 3)))
            else:
                ax.plot(x, y, label=str(metric), color=color)

            ax.fill_between(x, y - s, y + s, color=color, alpha=PLOT_CFG["shade_alpha"])

        ax.set_title(title, fontsize=14, weight="bold", style="italic")
        ax.set_xlabel(PLOT_CFG["xlabel"], fontsize=12)
        ax.grid(True, alpha=0.3)

    # y label on the first axis only
    g.axes.flat[0].set_ylabel(PLOT_CFG["ylabel"], fontsize=12)

    # ---- Global legend outside figure ----
    handles_labels = [ax.get_legend_handles_labels() for ax in g.axes.flat]
    handles = sum((hl[0] for hl in handles_labels), [])
    labels  = sum((hl[1] for hl in handles_labels), [])
    # Deduplicate (keep first occurrence)
    uniq = {}
    for h, lab in zip(handles, labels):
        if lab not in uniq:
            uniq[lab] = h

    if PLOT_CFG["legend_out"]:
        g.fig.legend(
            list(uniq.values()), list(uniq.keys()),
            loc=PLOT_CFG["legend_loc"],
            bbox_to_anchor=PLOT_CFG["legend_bbox"],
            ncol=PLOT_CFG["legend_ncol"],
            frameon=True,
            prop={"size": PLOT_CFG["legend_fontsize"]},
        )
        g.fig.subplots_adjust(right=PLOT_CFG["right_pad"])
    
    # Remove per-axis legends if present
    for ax in g.axes.flat:
        leg = ax.get_legend()
        if leg:
            leg.remove()

    out = outfile or PLOT_CFG["outfile"]
    g.fig.savefig(out, dpi=PLOT_CFG["dpi"], bbox_inches="tight", format="pdf")
    print(f"Figure saved as '{out}'")
    plt.show()

# =============================================================================
# 7) Execution: (1) Entity count / (2) Noise / (3) Skewness (Zipf alpha) -- save 3-panel PDF each
# =============================================================================
if __name__ == "__main__":
    setup_style()
    csv_path = "gpt2train/probe_results_multi.csv"
    stats_df = load_stats(csv_path)
    print(f"Stats: {stats_df.shape}")

    # (1) Entities (50k / 100k / 200k)
    models_entities = [
        'bioS_noise_0_multiple-context_250818-124537',       # 50k
        'bioS_noise_0_100k_multiple-context_250901-083541',  # 100k
        'bioS_noise_0_200k_multiple-context_250901-052022',  # 200k
    ]
    plot_with_std(stats_df, models_entities, outfile="entities_3panel.pdf")

    # (2) Noise (1% / 5% / 10%)
    models_noise = [
        'bioS_noise_1_multiple-context_passage-noise_250816-095848',
        'bioS_noise_5_multiple-context_passage-noise_250816-125421',
        'bioS_noise_10_multiple-context_passage-noise_250816-155036',
    ]
    plot_with_std(stats_df, models_noise, outfile="noise_3panel.pdf")

    # (3) Skewness (Zipf alpha = 0.5 / 1.0 / 2.0)
    models_skew = [
        'zip_text_aug_zipf_50_1_250901-114016',     # alpha = 0.5
        'zipf_textlist_out_zipf_100_1_250816-151409',  # alpha = 1.0
        'zip_text_aug_zipf_200_1_250901-160857',    # alpha = 2.0
    ]
    plot_with_std(stats_df, models_skew, outfile="skewness_3panel.pdf")